# Sections 8–10: KMeans Plots, Cluster Visualization & Business Recommendations
**Continuation of CBSOT_churnanalysis.ipynb** — run that notebook first, then run this one.

**Prerequisites (already computed in the main notebook):**
- `segmentation_data` — DataFrame with Tenure Months, Monthly Charges, Total Charges, Churn Probability
- `scaled_data` — StandardScaler-transformed version of the above
- `churn_probability` — array of RF churn probabilities
- `df_orig` — original raw Excel data

## 8. Elbow & Silhouette Plots

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

wcss, sil_scores = [], []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(scaled_data)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(scaled_data, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(K_range), wcss, marker='o', color='steelblue')
axes[0].set_title('Elbow Method (WCSS)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS')

axes[1].plot(list(K_range), sil_scores, marker='o', color='green')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = list(K_range)[sil_scores.index(max(sil_scores))]
print(f'Best k by silhouette: {best_k}')

## 9. Final KMeans Clustering (k=3) & Cluster Visualization

In [ ]:
# Final clustering with k=3 (adjust if needed based on elbow/silhouette)
K_FINAL = 3
km_final = KMeans(n_clusters=K_FINAL, random_state=42)
segmentation_data['Cluster'] = km_final.fit_predict(scaled_data)

# Auto-name clusters by mean churn probability
cp_rank = segmentation_data.groupby('Cluster')['Churn Probability'].mean().sort_values()
cluster_names = {}
for i, cluster_id in enumerate(cp_rank.index):
    if i == 0:
        cluster_names[cluster_id] = 'Loyal Low-Risk'
    elif i == len(cp_rank) - 1:
        cluster_names[cluster_id] = 'High-Risk'
    else:
        cluster_names[cluster_id] = f'Medium-Risk {i}'

segmentation_data['Segment'] = segmentation_data['Cluster'].map(cluster_names)

print('Cluster Summary:')
print(segmentation_data.groupby('Segment').mean().round(2))

In [ ]:
# Scatter: Monthly Charges vs Churn Probability, colored by segment
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['steelblue', 'salmon', 'green', 'orange']
for i, (seg, grp) in enumerate(segmentation_data.groupby('Segment')):
    axes[0].scatter(grp['Monthly Charges'], grp['Churn Probability'],
                    label=seg, alpha=0.4, s=8, color=colors[i % len(colors)])
axes[0].set_xlabel('Monthly Charges')
axes[0].set_ylabel('Churn Probability')
axes[0].set_title('Monthly Charges vs Churn Probability')
axes[0].legend(fontsize=8)

# Bar: customers per segment
seg_counts = segmentation_data['Segment'].value_counts()
seg_counts.plot(kind='bar', ax=axes[1], color=colors[:len(seg_counts)])
axes[1].set_title('Customers per Segment')
axes[1].set_ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Business Recommendations & Revenue at Risk

In [ ]:
segmentation_data['Revenue at Risk'] = (
    segmentation_data['Monthly Charges'] * segmentation_data['Churn Probability']
)

profile = segmentation_data.groupby('Segment').agg(
    Customers=('Segment', 'count'),
    Avg_Churn_Prob=('Churn Probability', 'mean'),
    Avg_Monthly=('Monthly Charges', 'mean'),
    Revenue_at_Risk=('Revenue at Risk', 'sum')
).round(2)

print('=== Segment Profiles ===')
print(profile)

# Revenue at risk bar chart
fig, ax = plt.subplots(figsize=(8, 4))
profile['Revenue_at_Risk'].sort_values().plot(kind='barh', ax=ax, color=['green', 'orange', 'red'])
ax.set_title('Monthly Revenue at Risk per Segment ($)')
ax.set_xlabel('Revenue at Risk ($)')
plt.tight_layout()
plt.show()

In [ ]:
print('\n=== Recommended Actions by Segment ===\n')

actions_map = {
    'High': [
        'Immediate personalised outreach (phone/email within 48h)',
        'Offer loyalty discount or contract upgrade',
        'Assign dedicated account manager'
    ],
    'Medium': [
        'Send satisfaction survey',
        'Offer add-on services (streaming, security)',
        'Enrol in loyalty / referral programme'
    ],
    'Loyal': [
        'Upsell to premium plans',
        'Referral incentives',
        'Reward with anniversary offers'
    ]
}

for seg in profile.index:
    cp = profile.loc[seg, 'Avg_Churn_Prob']
    n  = int(profile.loc[seg, 'Customers'])
    rev = profile.loc[seg, 'Revenue_at_Risk']

    if 'High' in seg:
        emoji, key = '🔴', 'High'
    elif 'Medium' in seg:
        emoji, key = '🟡', 'Medium'
    else:
        emoji, key = '🟢', 'Loyal'

    print(f'{emoji} {seg} — {n} customers | Churn Prob: {cp:.0%} | Revenue at Risk: ${rev:,.0f}')
    for action in actions_map[key]:
        print(f'   → {action}')
    print()

## 11. Export Model to `churn_app/data/` for Streamlit App

In [ ]:
import os, pickle

# Path relative to this notebook (adjust if running from a different directory)
EXPORT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'churn_app', 'data')
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save the tuned RF model
with open(os.path.join(EXPORT_DIR, 'rf_model.pkl'), 'wb') as f:
    pickle.dump(rf_tuned, f)

print(f'Model saved to {EXPORT_DIR}/rf_model.pkl')
print('Streamlit app can now load this model with pickle.load() instead of retraining.')

## 12. How to Launch the Streamlit App

After running all notebook cells, launch the dashboard:

```bash
cd churn_app
pip install -r requirements.txt          # first time only
streamlit run app.py
```

Then in the sidebar:
1. Upload `data/Telco_customer_churn.xlsx` (already copied there)
2. Select a model and number of segments
3. Click **🚀 Run Full Pipeline**

The app replicates every step in this notebook with an interactive UI.